# Example: Distributions and how to define new ones

The following notebook provides tutorial and code for defining distributions.

## Defining new metrics for parameter optimization

Hyperparameter optimization uses the Optuna library as a backend, and thus any distribution must be derived from its FloatDistribution (or BaseDistribution) class utilizing inverse transforms to map a uniform probability sample to the actual distribution value. It is a bit complicated, but if you follow the general structure of the original FloatDistribution/BaseDistribution, and the distributions already defined in garisom-tools (utils/distributions.py), it should be easy to implement your own.

In [6]:
from garisom_tools.optimization import get_mapping_dict

# Needed for new distribution declaration
from garisom_tools.utils.distributions import FloatDistribution
from scipy.stats import gamma

In [ ]:
# Define new distribution derived from Optuna FloatDistribution or BaseDistribution
class GammaDistribution(FloatDistribution):
    """
    Gamma distribution compatible with Optuna optimization.

    Uses inverse transform sampling to convert uniform random variables
    to gamma samples.

    Attributes:
        shape (float): Shape parameter (k).
        scale (float): Scale parameter (theta).
        low (float): Lower bound for internal uniform sampling (1e-8).
        high (float): Upper bound for internal uniform sampling (1-1e-8).
    """
    def __init__(self, shape=1.0, scale=1.0):
        """
        Initialize the gamma distribution.

        Args:
            shape (float, optional): Shape parameter k. Defaults to 1.0.
            scale (float, optional): Scale parameter theta. Defaults to 1.0.
        """
        self.shape = shape
        self.scale = scale
        self.low = 1e-8
        self.high = 1 - 1e-8
        super().__init__(self.low, self.high)

    def single(self) -> bool:
        """Check if distribution represents a single value."""
        return False

    def _contains(self, param_value_in_internal_repr):
        """Check if parameter is within valid range."""
        return isinstance(param_value_in_internal_repr, float) and self.low <= param_value_in_internal_repr <= self.high

    def _sample(self, rng):
        """Sample from the gamma distribution using inverse transform."""
        p = rng.uniform(self.low, self.high)
        return gamma.ppf(p, a=self.shape, scale=self.scale)

    def to_internal_repr(self, param_value_in_external_repr):
        """Transform gamma value to internal uniform representation."""
        p = gamma.cdf(param_value_in_external_repr, a=self.shape, scale=self.scale).item()
        return p

    def to_external_repr(self, param_value_in_internal_repr):
        """Transform internal uniform value to gamma value."""
        return gamma.ppf(param_value_in_internal_repr, a=self.shape, scale=self.scale)


In [8]:
# Add to the mapping dict
mappings = get_mapping_dict()
mappings['gamma'] = GammaDistribution

In [9]:
# Create a space config with new distribution
space_config = {
    'i_fieldCapFrac': ['uniform', [0.5, 1]],
    'i_fieldCapPercInit': ['uniform', [50, 100]],
    'i_rootBeta': ['uniform', [0.8, 1]],
    'i_leafAreaIndex': ['truncnorm', [3.7, 0.5, 1, 5]],
    'i_someParameter': ['gamma', [1, 1]]
}

# Then use this space_config in some OptimizationConfig

## Defining new metrics for Monte Carlo Simultations

It is much easier to define distributions for Monte Carlo simulations since the garisom-tools library directly calls a scipy distribution function ppf method, thus you can use any scipy distribution and add a function wrapper over it.

In [10]:
from garisom_tools.montecarlo import get_mapping_dict

# Import scipy distribution
from scipy.stats import gamma

In [ ]:
def get_scipy_gamma(shape=1.0, scale=1.0):
    return gamma(a=shape, scale=scale)

In [ ]:
# Add to the mapping dict
mappings = get_mapping_dict()
mappings['gamma'] = get_scipy_gamma

In [ ]:
# Create a space config with new distribution
space_config =  {  # SpaceConfig using scipy mappings defined in montecarlo/config.py
        "i_baperga": ["truncnorm", [43.61021, 2.78188]],  # truncnorm through scipy has default bounds
        "i_height": ["truncnorm", [1.586, 0.09]],
        "i_leafWidth": ["truncnorm", [0.03439, 0.00089]],
        "i_cr": ["uniform", [10.9202863, 12.3192219]],
        "i_br": ["uniform", [2.51387228, 2.79282904]],
        "i_cs": ["uniform", [10.9202863, 12.3192219]],
        "i_bs": ["uniform", [2.51387228, 2.79282904]],
        "i_cl": ["uniform", [10.9202863, 12.3192219]],
        "i_bl": ["uniform", [2.51387228, 2.79282904]],
        "i_vmax25": ["truncnorm", [93.877834, 3.85]],
        "i_jmax25": ["truncnorm", [156.78, 6.43]],
        "i_newParameter": ["gamma", [1.0, 1.0]]
    },

# Then use this space_config in some MonteCarloConfig